<a href="https://colab.research.google.com/github/lilthgh/Fraud-detection-using-Autoencoder/blob/main/Copy_of_Autoencoder%2Bsmote_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score
from sklearn.metrics import f1_score

from google.colab import files
uploaded = files.upload()




Saving creditcard.csv to creditcard.csv


In [2]:
# Load the data
df = pd.read_csv("creditcard.csv")

print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# 1. Check missing values
print("Missing values:\n", df.isnull().sum())

# 2. Check duplicates
print("Duplicate rows before:", df.duplicated().sum())

# 3. Remove duplicates
df = df.drop_duplicates()

# 4. Check duplicates again
print("Duplicate rows after:", df.duplicated().sum())
print("Shape after removing duplicates:", df.shape)

Missing values:
 Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64
Duplicate rows before: 1081
Duplicate rows after: 0
Shape after removing duplicates: (283726, 31)


In [4]:
# 5. Check class distribution
print("\nClass distribution:")
print(df['Class'].value_counts())

# 6. Check class percentage
print("\nClass percentage:")
print(df['Class'].value_counts(normalize=True) * 100)


Class distribution:
Class
0    283253
1       473
Name: count, dtype: int64

Class percentage:
Class
0    99.83329
1     0.16671
Name: proportion, dtype: float64


In [5]:
#scale data
rob_scaler = RobustScaler()
df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1, 1)) # 2D numpy
df['scaled_time'] = rob_scaler.fit_transform(df['Time'].values.reshape(-1, 1))
df.drop(['Time', 'Amount'], axis=1, inplace=True) # remove the original time & amount columns
scaled_amount = df['scaled_amount'] # save temp to rearranging column order later
scaled_time = df['scaled_time']
df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount) # add scaled_amount as the first column
df.insert(1, 'scaled_time', scaled_time) # add scaled_time as the second column
print(df.head())

   scaled_amount  scaled_time        V1        V2        V3        V4  \
0       1.774718    -0.995290 -1.359807 -0.072781  2.536347  1.378155   
1      -0.268530    -0.995290  1.191857  0.266151  0.166480  0.448154   
2       4.959811    -0.995279 -1.358354 -1.340163  1.773209  0.379780   
3       1.411487    -0.995279 -0.966272 -0.185226  1.792993 -0.863291   
4       0.667362    -0.995267 -1.158233  0.877737  1.548718  0.403034   

         V5        V6        V7        V8  ...       V20       V21       V22  \
0 -0.338321  0.462388  0.239599  0.098698  ...  0.251412 -0.018307  0.277838   
1  0.060018 -0.082361 -0.078803  0.085102  ... -0.069083 -0.225775 -0.638672   
2 -0.503198  1.800499  0.791461  0.247676  ...  0.524980  0.247998  0.771679   
3 -0.010309  1.247203  0.237609  0.377436  ... -0.208038 -0.108300  0.005274   
4 -0.407193  0.095921  0.592941 -0.270533  ...  0.408542 -0.009431  0.798278   

        V23       V24       V25       V26       V27       V28  Class  
0 -0.1104

In [6]:
# separate input features and target
X = df.drop('Class', axis=1) #all columns except class
y = df['Class'] #prediction 0 or 1
# split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
) # 80% train,reproducible,same ratio
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train class distribution:\n", y_train.value_counts())
print("y_test class distribution:\n", y_test.value_counts())

X_train shape: (226980, 30)
X_test shape: (56746, 30)
y_train class distribution:
 Class
0    226602
1       378
Name: count, dtype: int64
y_test class distribution:
 Class
0    56651
1       95
Name: count, dtype: int64


## Approach 1: SMOTE-based Autoencoder for Classification

This approach uses SMOTE to balance the classes in the training data, then trains an autoencoder on this balanced data. The encoded features are then used to train a traditional classifier (Logistic Regression in this case) to predict fraud.

In [11]:
# Apply SMOTE on training data only
smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_res.value_counts())

# Convert SMOTE data to NumPy
X_train_res_np = X_train_res.values
y_train_res_np = y_train_res.values

X_test_np = X_test.values
y_test_np = y_test.values

Before SMOTE:
Class
0    226602
1       378
Name: count, dtype: int64

After SMOTE:
Class
0    226602
1    226602
Name: count, dtype: int64


In [12]:
#Building new SMOTE Autoencoder
input_dim = X_train_res_np.shape[1]

input_layer_smote = Input(shape=(input_dim,))

encoded_smote = Dense(16, activation='relu')(input_layer_smote)
encoded_smote = Dense(8, activation='relu', name='latent_layer_smote')(encoded_smote)

decoded_smote = Dense(16, activation='relu')(encoded_smote)
decoded_smote = Dense(input_dim, activation='linear')(decoded_smote)

autoencoder_smote = Model(inputs=input_layer_smote, outputs=decoded_smote)

autoencoder_smote.compile(optimizer='adam', loss='mse')

autoencoder_smote.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent_layer_smote (Dense)      │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 30)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,286 (5.02 KB)

 Trainable params: 1,286 (5.02 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:

#earlystop


early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Training SMOTE Autoencoder
history_smote = autoencoder_smote.fit(
    X_train_res_np,
    X_train_res_np,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - loss: 1.7704 - val_loss: 0.7179
Epoch 2/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 0.5226 - val_loss: 0.5011
Epoch 3/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.4308 - val_loss: 0.4413
Epoch 4/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.4051 - val_loss: 0.4188
Epoch 5/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.3949 - val_loss: 0.4003
Epoch 6/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3850 - val_loss: 0.3835
Epoch 7/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3755 - val_loss: 0.3652
Epoch 8/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.3648 - val_loss: 0.3574
Epoch 9/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3573 - val_loss: 0.3462
Epoch 10/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3495 - val_loss: 0.3396
Epoch 11/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.3435 - val_loss: 0.3366
Epoch 12/20
1417/1417 ━━━━━━━

In [15]:
#traning with smote data
history_smote = autoencoder_smote.fit(
    X_train_res_np,
    X_train_res_np,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3235 - val_loss: 0.3261
Epoch 2/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.3214 - val_loss: 0.3244
Epoch 3/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3194 - val_loss: 0.3218
Epoch 4/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.3182 - val_loss: 0.3210
Epoch 5/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.3173 - val_loss: 0.3231
Epoch 6/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3161 - val_loss: 0.3174
Epoch 7/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3152 - val_loss: 0.3218
Epoch 8/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.3144 - val_loss: 0.3171
Epoch 9/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.3135 - val_loss: 0.3141
Epoch 10/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.3126 - val_loss: 0.3153
Epoch 11/20
1417/1417 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.3123 - val_loss: 0.3190
Epoch 12/20
1417/1417 ━━━━━━━━

### Note on Autoencoder Training

The previous cell (`wAhWxOxz7iKS`) already trained the `autoencoder_smote` model with early stopping. Running the training again in this cell (`d1xyLJyH6Orw`) would effectively continue training from the state of the best weights from the *first* training. For clarity and to avoid redundant steps, we'll consider the model sufficiently trained after the first run. The next step will proceed with using the trained `autoencoder_smote` to extract features.

---

In [16]:
#Extract encoded features
encoder_smote = Model(
    inputs=autoencoder_smote.input,
    outputs=autoencoder_smote.get_layer('latent_layer_smote').output
)

X_train_encoded_smote = encoder_smote.predict(X_train_res_np)
X_test_encoded_smote = encoder_smote.predict(X_test_np)

14163/14163 ━━━━━━━━━━━━━━━━━━━━ 18s 1ms/step
1774/1774 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [17]:
#train classifier
from sklearn.linear_model import LogisticRegression

clf_smote = LogisticRegression(max_iter=1000, random_state=42)

clf_smote.fit(X_train_encoded_smote, y_train_res_np)

LogisticRegression(max_iter=1000, random_state=42)

In [20]:
#evaluate
y_pred_smote = clf_smote.predict(X_test_encoded_smote)
y_proba_smote = clf_smote.predict_proba(X_test_encoded_smote)[:, 1]

print("Confusion Matrix:\n", confusion_matrix(y_test_np, y_pred_smote))
print("\nClassification Report:\n", classification_report(y_test_np, y_pred_smote))
print("\nROC-AUC:", roc_auc_score(y_test_np, y_proba_smote))
print("PR-AUC:", average_precision_score(y_test_np, y_proba_smote))

Confusion Matrix:
 [[55095  1556]
 [   13    82]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.97      0.99     56651
           1       0.05      0.86      0.09        95

    accuracy                           0.97     56746
   macro avg       0.52      0.92      0.54     56746
weighted avg       1.00      0.97      0.98     56746


ROC-AUC: 0.9664195085514354
PR-AUC: 0.5186320871344974


## Approach 2: Anomaly Detection with Autoencoder

This approach trains an autoencoder exclusively on *normal* transactions. The idea is that the autoencoder will learn to reconstruct normal patterns effectively, but will struggle to reconstruct anomalous (fraudulent) transactions, leading to a higher reconstruction error for anomalies. A threshold on this reconstruction error is then used to classify transactions as normal or fraudulent.

In [21]:
# keep only normal transactions for autoencoder training
X_train_normal = X_train[y_train == 0] #non-fraud for traning

print("Original X_train shape:", X_train.shape)
print("Normal-only X_train shape:", X_train_normal.shape)
#convert to numpy
X_train_normal_np = X_train_normal.values
X_test_np = X_test.values
y_test_np = y_test.values

Original X_train shape: (226980, 30)
Normal-only X_train shape: (226602, 30)


In [22]:
# Autoencode
# input dimension
input_dim = X_train_normal_np.shape[1] #number of the columns

# input layer in neyral network/starting point
input_layer = Input(shape=(input_dim,))
# encoder
encoded = Dense(16, activation='relu')(input_layer)# reduce demention from 30 to 16
encoded = Dense(8, activation='relu')(encoded)# reduce 16 to 8 bottleneck layer

# begining of decoder
decoded = Dense(16, activation='relu')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded) #real numeric values

# build model
autoencoder = Model(inputs=input_layer, outputs=decoded) #conects Input → Encoder → Decoder → Output

# compile model
autoencoder.compile(optimizer='adam', loss='mse') #adam=How the model learns (updates weights),mse=Measures error

# show model summary
autoencoder.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 30)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,286 (5.02 KB)

 Trainable params: 1,286 (5.02 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:

early_stop = EarlyStopping(
    monitor='val_loss',#validation loss
    patience=3,#if 3 epochs in a row stop
    restore_best_weights=True
)#early stopping rule Keras keeps checking model performance if it stops improving, training stops automatically
#Traning Autoencoder
history = autoencoder.fit(
    X_train_normal_np,#input
    X_train_normal_np,#the target output
    epochs=20,#train for up to 20 epochs
    batch_size=256,#Splits training data into smaller chunks of 256 rows
    validation_split=0.2,
    shuffle=True,#learn more robustly
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.7339 - val_loss: 0.4807
Epoch 2/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4401 - val_loss: 0.4217
Epoch 3/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.4044 - val_loss: 0.3976
Epoch 4/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.3884 - val_loss: 0.3831
Epoch 5/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3763 - val_loss: 0.3738
Epoch 6/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3682 - val_loss: 0.3693
Epoch 7/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3613 - val_loss: 0.3599
Epoch 8/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.3532 - val_loss: 0.3547
Epoch 9/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.3469 - val_loss: 0.3478
Epoch 10/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3412 - val_loss: 0.3429
Epoch 11/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3380 - val_loss: 0.3401
Epoch 12/20
709/709 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

In [24]:
#calculating threshold
X_train_pred = autoencoder.predict(X_train_normal_np)#how well are model works
train_error = np.mean(np.power(X_train_normal_np - X_train_pred, 2), axis=1)#calculates avg mean of (square the diffrences between pred & normal) for each row

threshold = np.percentile(train_error, 95)#95% <threshold
print("Threshold:", threshold)

7082/7082 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step
Threshold: 0.6794112740322045


In [25]:
#Test reconstruction error
X_test_pred = autoencoder.predict(X_test_np)
reconstruction_error = np.mean(np.power(X_test_np - X_test_pred, 2), axis=1)

1774/1774 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [26]:
#Predict fraud
#new table for error & real labels
error_df = pd.DataFrame({
    'reconstruction_error': reconstruction_error,
    'true_class': y_test_np
})

y_pred_autoencoder = (error_df['reconstruction_error'] > threshold).astype(int)#compare error with threshold & convert T/F to 1/0

error_df.head()

,reconstruction_error,true_class
0,0.085366,0
1,0.131984,0
2,0.291104,0
3,0.244129,0
4,0.272702,0


In [27]:
#Evaluate
print("Confusion Matrix:\n", confusion_matrix(y_test_np, y_pred_autoencoder))
print("\nClassification Report:\n", classification_report(y_test_np, y_pred_autoencoder))
print("\nROC-AUC:", roc_auc_score(y_test_np, reconstruction_error))
print("PR-AUC:", average_precision_score(y_test_np, reconstruction_error))

Confusion Matrix:
 [[53836  2815]
 [   17    78]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.95      0.97     56651
           1       0.03      0.82      0.05        95

    accuracy                           0.95     56746
   macro avg       0.51      0.89      0.51     56746
weighted avg       1.00      0.95      0.97     56746


ROC-AUC: 0.9543348052573049
PR-AUC: 0.29314808416078414


In [28]:
#f1 optimizer calculting best threshold
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.linspace(reconstruction_error.min(), reconstruction_error.max(), 200)#min<Creates200 equally spaced numbers threshold<max


best_threshold = 0
best_f1 = 0
best_precision = 0
best_recall = 0

for t in thresholds:
    y_pred_temp = (reconstruction_error > t).astype(int)

    current_f1 = f1_score(y_test_np, y_pred_temp)
    current_precision = precision_score(y_test_np, y_pred_temp, zero_division=0)
    current_recall = recall_score(y_test_np, y_pred_temp)

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = t
        best_precision = current_precision
        best_recall = current_recall

print("Best threshold:", best_threshold)
print("Best F1-score:", best_f1)
print("Best precision:", best_precision)
print("Best recall:", best_recall)

Best threshold: 7.2196520048394826
Best F1-score: 0.4112903225806452
Best precision: 0.3333333333333333
Best recall: 0.5368421052631579


### Anomaly Detection Autoencoder: Evaluation with Optimized F1-score Threshold

This evaluation uses the `best_threshold` calculated from the `reconstruction_error` to classify anomalies. Note that this is for the *Anomaly Detection* approach, which trained its autoencoder only on normal transactions, and does not directly involve SMOTE in the prediction step.

In [30]:
# Evaluate with the best threshold found for the Anomaly Detection Autoencoder
y_pred_autoencoder_f1_optimized = (reconstruction_error > best_threshold).astype(int)

print("Confusion Matrix (Anomaly Detection with Optimized F1 Threshold):\n", confusion_matrix(y_test_np, y_pred_autoencoder_f1_optimized))
print("\nClassification Report (Anomaly Detection with Optimized F1 Threshold):\n", classification_report(y_test_np, y_pred_autoencoder_f1_optimized))
print("\nROC-AUC (Anomaly Detection):", roc_auc_score(y_test_np, reconstruction_error))
print("PR-AUC (Anomaly Detection):", average_precision_score(y_test_np, reconstruction_error))

Confusion Matrix (Anomaly Detection with Optimized F1 Threshold):
 [[56549   102]
 [   44    51]]

Classification Report (Anomaly Detection with Optimized F1 Threshold):
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.33      0.54      0.41        95

    accuracy                           1.00     56746
   macro avg       0.67      0.77      0.71     56746
weighted avg       1.00      1.00      1.00     56746


ROC-AUC (Anomaly Detection): 0.9543348052573049
PR-AUC (Anomaly Detection): 0.29314808416078414


## Re-evaluating SMOTE-based Autoencoder for Classification

To ensure we have a clear view of the SMOTE model's performance as the final output, here are its evaluation metrics once more. These results come from the Logistic Regression classifier trained on features extracted by the autoencoder, which itself was trained on SMOTE-resampled data.

In [31]:
# Evaluate the Logistic Regression classifier (clf_smote) trained on SMOTE-encoded features
y_pred_smote_final = clf_smote.predict(X_test_encoded_smote)
y_proba_smote_final = clf_smote.predict_proba(X_test_encoded_smote)[:, 1]

print("Confusion Matrix (SMOTE-based Autoencoder + Classifier):\n", confusion_matrix(y_test_np, y_pred_smote_final))
print("\nClassification Report (SMOTE-based Autoencoder + Classifier):\n", classification_report(y_test_np, y_pred_smote_final))
print("\nROC-AUC (SMOTE-based Autoencoder + Classifier):", roc_auc_score(y_test_np, y_proba_smote_final))
print("PR-AUC (SMOTE-based Autoencoder + Classifier):", average_precision_score(y_test_np, y_proba_smote_final))

Confusion Matrix (SMOTE-based Autoencoder + Classifier):
 [[55095  1556]
 [   13    82]]

Classification Report (SMOTE-based Autoencoder + Classifier):
               precision    recall  f1-score   support

           0       1.00      0.97      0.99     56651
           1       0.05      0.86      0.09        95

    accuracy                           0.97     56746
   macro avg       0.52      0.92      0.54     56746
weighted avg       1.00      0.97      0.98     56746


ROC-AUC (SMOTE-based Autoencoder + Classifier): 0.9664195085514354
PR-AUC (SMOTE-based Autoencoder + Classifier): 0.5186320871344974
